# __SLUG__: baseline

Goal: a simple model with a trustworthy CV score that every later idea must beat. Change `make_splits` if EDA shows the data has groups or a time order.

In [ ]:
import re

import pandas as pd
from xgboost import XGBClassifier, XGBRegressor

from kagglekit import load_project, log_experiment, make_splits, make_submission, run_cv, seed_everything

SEED = 42
seed_everything(SEED)

project = load_project()
train = project.read_train()
test = project.read_test()
target = project.target_column()
id_name = project.id_name()

In [ ]:
def prepare(df):
    """Keep numeric, boolean and low-cardinality text columns."""
    df = df.drop(columns=[target, id_name], errors="ignore")
    text = [c for c in df.select_dtypes(exclude=["number", "bool"]).columns if train[c].nunique() <= 50]
    df = df[df.select_dtypes(include=["number", "bool"]).columns.tolist() + text].copy()
    for column in text:
        categories = sorted(train[column].dropna().map(str).unique())
        df[column] = pd.Categorical(df[column].map(str, na_action="ignore"), categories=categories)
    # XGBoost rejects feature names containing [, ] or <.
    return df.rename(columns=lambda name: re.sub(r"[\[\]<]", "", name))


X = prepare(train)
X_test = prepare(test)
y = train[target]
X.dtypes.value_counts()

In [ ]:
params = {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5, "subsample": 0.8, "colsample_bytree": 0.8}
model_class = XGBClassifier if project.task == "binary" else XGBRegressor


def make_model():
    return model_class(**params, enable_categorical=True, random_state=SEED)


strategy = "stratified" if project.task == "binary" else "kfold"
splits = make_splits(y, strategy=strategy, seed=SEED)
result = run_cv(make_model, X, y, splits, project.metric, project.task, X_test=X_test)
result

In [ ]:
submission = make_submission(project, result.test_pred, "baseline-xgb", ids=test[id_name])
log_experiment(project, "baseline-xgb", result, params=params, submission=submission)
submission